In [1]:
## Classification

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


# feature selection
def sfs_forwardSelection(indep_x,dep_y,model,n):
    sfs=SequentialFeatureSelector(estimator=model, n_features_to_select=n, direction='forward', scoring='accuracy', cv=5)
    sfs.fit(indep_x,dep_y)
    selected_features = indep_x.columns[sfs.get_support()]
    return selected_features
    
def sfs_backwardElimination(indep_x,dep_y,model,n):
    sfs=SequentialFeatureSelector(estimator=model, n_features_to_select=n, direction='backward', scoring='accuracy', cv=5)
    sfs.fit(indep_x,dep_y)
    selected_features = indep_x.columns[sfs.get_support()]
    return selected_features
    
# tran test split and standardization 
def split_scaler(indep_x,dep_y):
    x_train,x_test,y_train,y_test = train_test_split(indep_x,dep_y,test_size=0.20,random_state=0)
    # standardization
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return  x_train,x_test,y_train,y_test

# model prediction & evaluation 
def cm_pred_eval(classifier,x_test):
    # prediction
    y_pred=classifier.predict(x_test)
    # confusion matrix
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    # classification report 
    from sklearn.metrics import classification_report
    classi_report=classification_report(y_test,y_pred)
    # accuracy 
    from sklearn.metrics import accuracy_score
    accuracy=accuracy_score(y_test,y_pred)
    return classifier,accuracy,cm,classi_report,x_test,y_test

### --------------- classification algortihms -----------------------------
# logistic regression 
def logistic(x_train,y_train,x_test):
    # model creation
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# KNN 
def knn(x_train,y_train,x_test):
    # model creation
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski')
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - linear 
def svm_linear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='linear',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - Non linear 
def svm_nonlinear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# navie bayes
def naive(x_train,y_train,x_test):
    # model creation 
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# decision tree
def decision_tree(x_train,y_train,x_test):
     # model creation 
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# random forest
def random_forest(x_train,y_train,x_test):
     # model creation 
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test




In [2]:


data_original = pd.read_csv('preprocessed_CKD.csv')
data_ckd= data_original
data_ckd

data_ckd=pd.get_dummies(data_ckd,drop_first=True,dtype=int)
data_ckd.head()

# ip op split 
indep_x=data_ckd.drop('classification_notckd',axis=1)
dep_y = data_ckd['classification_notckd']

# k best 
no_of_features = 5


x_train,x_test,y_train,y_test=split_scaler(indep_x,dep_y)   # (selected important features , target variable) 

models = {
    'Decision Tree': DecisionTreeClassifier(criterion='entropy',random_state=0),
    # 'Random Forest': RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0),
    # 'Logistic Regression': LogisticRegression(random_state=0),
    # 'svml': SVC(kernel='linear',random_state=0),
    # 'svmnl': SVC(kernel='rbf',random_state=0),
    # 'knn':KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski'),
    # 'naive':GaussianNB()
}

# FORWARD SELECTION 
forward_results={}
for model_name,model in models.items():
    selected_features = sfs_forwardSelection(indep_x,dep_y,model,no_of_features)
    x_train,x_test,y_train,y_test=split_scaler(indep_x[selected_features],dep_y) 
    # Train model on selected features
    # print(selected_features)
    print(selected_features.tolist())
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    # Store results
    forward_results[model_name] = {
        'selected_features': selected_features.tolist(),
        'accuracy': accuracy_score(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred)
    }

pd.DataFrame(forward_results)

['sg', 'al', 'hemo', 'rc', 'htn_yes']


,Decision Tree
selected_features,"[sg, al, hemo, rc, htn_yes]"
accuracy,1.0
classification_report,precision recall f1-score ...
